# 🏆 Notebook 05 — Penentuan Model Terbaik & Ringkasan Presentasi
Skoring multi-kriteria menggunakan seluruh data dari jurnal.

> ✅ 100% berjalan di MacBook Air M2.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

t3  = pd.read_csv('../dataset/table3_timing_metrics.csv')
t9  = pd.read_csv('../dataset/table9_text_metrics_50prompts.csv')
t13 = pd.read_csv('../dataset/table13_code_metrics.csv')
sv  = pd.read_csv('../dataset/section47_syntactic_validity.csv')
print("✅ Semua data dimuat")

## 1. Bangun tabel multi-kriteria

In [ ]:
int8 = t3[t3.Precision=='INT8'].copy()

# VRAM estimasi dari ukuran model (dari jurnal Section 4)
vram = {'GPT-2':0.3, 'LLaMA-2-7B-Chat':7.5, 'Qwen1.5-1.8B-Chat':2.0}
int8['VRAM_GB'] = int8.Model.map(vram)

# CPU offload hanya LLaMA-2 RTX4070 FP16 (catatan Table 3)
int8['No_CPU_offload'] = ~((int8.Model=='LLaMA-2-7B-Chat')&(int8.GPU=='RTX4070'))
int8['No_CPU_offload'] = int8['No_CPU_offload'].astype(int)

# ROUGE-1 dari Table 9 (LLaMA-2 & Qwen) dan estimasi GPT-2 dari Table 6
rouge_map = {(r.GPU,r.Model): r.ROUGE1_mean for _,r in t9.iterrows()}
int8['ROUGE1'] = int8.apply(lambda r: rouge_map.get((r.GPU,r.Model), 0.50), axis=1)
bleu_map  = {(r.GPU,r.Model): r.BLEU_mean  for _,r in t9.iterrows()}
int8['BLEU']   = int8.apply(lambda r: bleu_map.get( (r.GPU,r.Model), 0.11), axis=1)

df_mc = int8[['GPU','Model','Tokens_per_s','BLEU','ROUGE1','VRAM_GB','No_CPU_offload']].copy()

print("=== Tabel Multi-Kriteria (INT8 konfigurasi) ===")
print(df_mc.to_string(index=False))

## 2. Normalisasi & scoring

In [ ]:
def norm(s, inv=False):
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series([1.0]*len(s), index=s.index)
    n = (s - mn) / (mx - mn)
    return 1-n if inv else n

# Bobot kriteria (total = 1.0)
W = {'tok':0.25, 'bleu':0.15, 'rouge1':0.25, 'vram':0.20, 'offload':0.15}

df_mc['n_tok']     = norm(df_mc.Tokens_per_s)
df_mc['n_bleu']    = norm(df_mc.BLEU)
df_mc['n_rouge1']  = norm(df_mc.ROUGE1)
df_mc['n_vram']    = norm(df_mc.VRAM_GB, inv=True)
df_mc['n_offload'] = df_mc.No_CPU_offload.astype(float)

df_mc['SKOR_TOTAL'] = (
    df_mc.n_tok    * W['tok'] +
    df_mc.n_bleu   * W['bleu'] +
    df_mc.n_rouge1 * W['rouge1'] +
    df_mc.n_vram   * W['vram'] +
    df_mc.n_offload* W['offload']
).round(4)

ranking = df_mc[['GPU','Model','Tokens_per_s','BLEU','ROUGE1','VRAM_GB','SKOR_TOTAL']]    .sort_values('SKOR_TOTAL', ascending=False).reset_index(drop=True)

print("=== RANKING MODEL (Multi-Criteria Scoring) ===")
print(ranking.to_string(index=False))
print()
print("Bobot kriteria:")
for k,v in W.items():
    print(f"  {k:12s}: {v:.0%}")

## 3. Visualisasi ranking

In [ ]:
labels = [f"{r.Model.split('-')[0]}
{r.GPU}" for _,r in ranking.iterrows()]
scores = ranking.SKOR_TOTAL.values
colors = ['#2ECC71' if i==0 else '#3498DB' if i==1 else '#95A5A6' for i in range(len(scores))]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels[::-1], scores[::-1], color=colors[::-1],
               edgecolor='white', linewidth=0.8)
for bar, score in zip(bars, scores[::-1]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Skor Total (Multi-Criteria)', fontsize=12)
ax.set_title('Ranking Model INT8 — Multi-Criteria Scoring\n'
             'Berdasarkan data Oprea & Bâra (2026)', fontweight='bold')
ax.set_xlim(0, 1.0)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
legend_patches = [mpatches.Patch(color='#2ECC71',label='🥇 Terbaik'),
                  mpatches.Patch(color='#3498DB',label='🥈 Runner-up'),
                  mpatches.Patch(color='#95A5A6',label='Lainnya')]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES/'model_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: outputs/figures/model_ranking.png")

## 4. Trade-off scatter plot (Speedup vs Quality)

In [ ]:
spd = pd.read_csv('../dataset/findings_summary.csv')
spd = spd[spd.Category=='Speedup'].copy()
spd['speedup_val'] = pd.to_numeric(spd.Value.str.replace('x',''), errors='coerce')

config_bleu = {'GPT2_RTX4070':0.11,'GPT2_RTX4080':0.11,
               'LLaMA2_RTX4070':0.18,'LLaMA2_RTX4080':0.12,
               'Qwen_RTX4070':0.13,'Qwen_RTX4080':0.11}
config_r1   = {'GPT2_RTX4070':0.50,'GPT2_RTX4080':0.50,
               'LLaMA2_RTX4070':0.51,'LLaMA2_RTX4080':0.52,
               'Qwen_RTX4070':0.62,'Qwen_RTX4080':0.39}
color_map   = {'GPT2':'#378ADD','LLaMA2':'#D85A30','Qwen':'#1D9E75'}

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, ykey, ylabel, ytitle in [
    (axes[0], config_bleu, 'BLEU Score', 'BLEU'),
    (axes[1], config_r1,   'ROUGE-1 Score', 'ROUGE-1')]:

    for cfg, row in spd.iterrows():
        name = row.Metric
        spd_v = min(row.speedup_val, 15)
        qual  = ykey.get(name, 0)
        prefix= name.split('_')[0]
        col   = color_map.get(prefix,'gray')
        ax.scatter(spd_v, qual, s=150, color=col, edgecolors='black', lw=0.8, zorder=3)
        ax.annotate(name.replace('_','
'), (spd_v, qual),
                    xytext=(5,5), textcoords='offset points', fontsize=7.5)

    ax.set_xlabel('Speedup INT8 vs FP16 (capped 15x)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f'Trade-off: Speedup vs {ytitle}', fontweight='bold')
    ax.grid(alpha=0.3)

patches = [mpatches.Patch(color='#378ADD',label='GPT-2'),
           mpatches.Patch(color='#D85A30',label='LLaMA-2'),
           mpatches.Patch(color='#1D9E75',label='Qwen1.5')]
axes[0].legend(handles=patches, fontsize=9)
plt.suptitle('Efficiency vs Quality Trade-off\nOprea & Bâra (2026)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES/'tradeoff_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: outputs/figures/tradeoff_scatter.png")

## 5. Klasifikasi model per use case

In [ ]:
print("=" * 65)
print("  KLASIFIKASI MODEL BERDASARKAN DATA JURNAL")
print("=" * 65)

use_cases = [
    ("Edge / IoT / Real-time",
     "Qwen1.5-1.8B INT8",
     "VRAM 2GB · 21-23 tok/s · No offload · ROUGE-1 0.618"),
    ("Kualitas Output Tertinggi",
     "LLaMA-2-7B FP16 RTX4080",
     "Depth & coherence terbaik (Table 4) · 50s inference"),
    ("Prototyping & Edukasi",
     "GPT-2 INT8",
     "Sangat ringan · MIT License · Setup cepat"),
    ("Balanced Quality + Speed",
     "LLaMA-2-7B INT8 RTX4080",
     "1.88x speedup · ROUGE-L 0.409 · Full GPU residency"),
]

for uc, model, reason in use_cases:
    print(f"\n  USE CASE : {uc}")
    print(f"  MODEL    : {model}")
    print(f"  ALASAN   : {reason}")

print()
print("=" * 65)

## 6. Kesimpulan final untuk presentasi

In [ ]:
best = ranking.iloc[0]
print("=" * 65)
print("  KESIMPULAN PENELITIAN (Oprea & Bâra, 2026)")
print("=" * 65)
print()
print("  1. INT8 PTQ rata-rata 3.4x lebih cepat dari FP16")
print("     (jurnal abstract — excl. CPU offload configs)")
print()
print("  2. Degradasi kualitas moderat dan konsisten:")
print("     BLEU turun ~0.10-0.18 range pada INT8")
print("     ROUGE-1 tetap cukup tinggi (0.39-0.62)")
print()
print("  3. Code generation lebih sensitif:")
print("     LLaMA-2 INT8: 93% syntactically valid")
print("     Qwen INT8   : 87% syntactically valid")
print()
print("  4. Degradasi utama = gaya/struktur, BUKAN kegagalan semantik")
print()
print("─" * 65)
print(f"  MODEL TERBAIK UNTUK DEPLOYMENT:")
print(f"  {best.Model} INT8 pada {best.GPU}")
print(f"  Skor multi-kriteria : {best.SKOR_TOTAL}")
print(f"  ROUGE-1             : {best.ROUGE1}")
print(f"  Throughput          : {best.Tokens_per_s} tok/s")
print(f"  VRAM                : {best.VRAM_GB} GB")
print(f"  Lisensi             : Apache 2.0")
print("─" * 65)